# Lesson 02b — NER & Extraction Pipelines
**Domain:** Legal & Contracts  
**Dataset:** Synthetic contract clauses (LLM-generated on-the-fly)  
**Time estimate:** ~2 h

## Learning objectives
- Extract: parties, monetary amounts, dates, jurisdictions, obligations
- LLM NER vs spaCy `en_core_web_lg` — when to use which
- Batch extraction with `asyncio` + error handling
- Partial output handling: `Optional` fields, post-hoc validation

In [ ]:
import sys, asyncio
sys.path.insert(0, "..")

from typing import Optional
from pydantic import BaseModel, Field, field_validator, ValidationError
import instructor
from openai import OpenAI, AsyncOpenAI

from llm_checker import check, hint, evaluate, progress

local_client  = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
async_client  = AsyncOpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
ic_client     = instructor.from_openai(local_client)
ic_async      = instructor.from_openai(async_client)
MODEL         = "local-model"

def raw_llm(prompt, max_tokens=400):
    r = local_client.chat.completions.create(
        model=MODEL, messages=[{"role":"user","content":prompt}],
        max_tokens=max_tokens, temperature=0.7
    )
    return r.choices[0].message.content.strip()

print("✅ Setup complete")

## Generate synthetic contract clauses

In [ ]:
print("Generating 50 synthetic contract clauses…")

CLAUSE_TEMPLATES = [
    "Write a {clause_type} clause for a {contract_type} contract between {party1} and {party2}. Include specific obligations, deadlines, and monetary amounts where appropriate.",
]

import random
CLAUSE_TYPES = ["payment", "liability", "termination", "NDA", "IP ownership", "indemnification", "dispute resolution"]
CONTRACT_TYPES = ["software licensing", "employment", "vendor services", "partnership", "SaaS subscription"]
COMPANIES = ["TechCorp Ltd", "DataSystems Inc", "CloudVentures GmbH", "NovaTech SA", "ByteWorks LLC",
             "AlphaServices Sp. z o.o.", "BetaCloud AG", "Nexus Solutions Ltd"]

random.seed(42)
synthetic_clauses = []
for i in range(50):
    clause_type   = random.choice(CLAUSE_TYPES)
    contract_type = random.choice(CONTRACT_TYPES)
    party1, party2 = random.sample(COMPANIES, 2)
    prompt = CLAUSE_TEMPLATES[0].format(
        clause_type=clause_type, contract_type=contract_type, party1=party1, party2=party2
    )
    clause = raw_llm(prompt, max_tokens=200)
    synthetic_clauses.append(clause)
    if (i + 1) % 10 == 0:
        print(f"  Generated {i+1}/50 clauses")

print(f"✅ {len(synthetic_clauses)} synthetic clauses generated")
print("\nSample clause:")
print(synthetic_clauses[0][:300])

---
## 🔵 EXAMPLE — Ex 02b-1: Legal entity extraction

Define `LegalEntities` and extract from 5 synthetic contract snippets.

In [ ]:
class LegalEntities(BaseModel):
    parties: list[str] = Field(min_length=2, description="List of contracting parties (≥ 2)")
    monetary_amounts: list[str] = Field(default_factory=list, description="All monetary amounts as strings")
    jurisdiction: Optional[str] = None
    effective_date: Optional[str] = None
    governing_law: Optional[str] = None

    @field_validator("parties")
    @classmethod
    def at_least_two_parties(cls, v):
        if len(v) < 2:
            raise ValueError("Must have at least 2 parties")
        return v

results_02b1 = []
for clause in synthetic_clauses[:5]:
    result = ic_client.chat.completions.create(
        model=MODEL,
        response_model=LegalEntities,
        messages=[{"role": "user", "content": f"Extract legal entities from:\n\n{clause}"}],
        max_retries=2,
    )
    results_02b1.append(result)
    print(f"Parties: {result.parties}  |  Amounts: {result.monetary_amounts}")

check([
    (all(len(r.parties) >= 2 for r in results_02b1),      "All extractions have ≥ 2 parties"),
    (all(isinstance(r.monetary_amounts, list) for r in results_02b1), "monetary_amounts is list"),
    (len(results_02b1) == 5,                               "5 clauses extracted without crash"),
], exercise_id="02b-1")

---
## 🟡 EXERCISE — Ex 02b-2: Async batch extraction with error handling

Write `extract_batch(clauses: list[str]) -> list[LegalEntities | None]`.
Use `asyncio.gather(return_exceptions=True)`.
If extraction fails → insert `None`. Run on all 50 clauses. Report failure rate.

**Auto-check verifies:**
- Returns list of same length as input
- Failures don't stop the batch
- Failure rate printed as %

In [ ]:
async def _extract_one(clause: str) -> LegalEntities:
    return await ic_async.chat.completions.create(
        model=MODEL,
        response_model=LegalEntities,
        messages=[{"role": "user", "content": f"Extract legal entities from:\n\n{clause}"}],
        max_retries=2,
    )

async def extract_batch(clauses: list) -> list:
    # TODO: use asyncio.gather(return_exceptions=True)
    # For each result: if isinstance(result, Exception) → append None
    raise NotImplementedError("Implement extract_batch()")

In [ ]:
async def run_batch_check():
    try:
        batch_results = await extract_batch(synthetic_clauses)
        n_ok    = sum(1 for r in batch_results if r is not None)
        n_fail  = sum(1 for r in batch_results if r is None)
        fail_pct = n_fail / len(synthetic_clauses) * 100

        print(f"Batch results: {n_ok} success, {n_fail} failures ({fail_pct:.1f}% failure rate)")

        check([
            (len(batch_results) == len(synthetic_clauses), "Output length == input length"),
            (n_ok > 0,                                     "At least some succeeded"),
            (isinstance(fail_pct, float),                  "Failure rate computed"),
        ], exercise_id="02b-2")
    except NotImplementedError:
        print("⚠️  Implement extract_batch() first.")

await run_batch_check()

---
## 🔴 CHALLENGE — Ex 02b-3: LLM NER vs spaCy comparison

Run spaCy (`en_core_web_lg`) and LLM NER on 20 contract clauses.
Compare: which entities spaCy detects correctly, where LLM is better.
Document 3 examples where each approach wins.

In [ ]:
try:
    import spacy
    nlp = spacy.load("en_core_web_lg")
    SPACY_AVAILABLE = True
    print("✅ spaCy en_core_web_lg loaded")
except Exception as e:
    print(f"⚠️  spaCy not available: {e}")
    print("   Run: python -m spacy download en_core_web_lg")
    SPACY_AVAILABLE = False

In [ ]:
if SPACY_AVAILABLE:
    comparison_results = []
    for clause in synthetic_clauses[:20]:
        # spaCy NER
        doc = nlp(clause)
        spacy_ents = [(ent.text, ent.label_) for ent in doc.ents]

        # LLM NER
        llm_result = ic_client.chat.completions.create(
            model=MODEL,
            response_model=LegalEntities,
            messages=[{"role": "user", "content": f"Extract entities:\n\n{clause}"}],
            max_retries=2,
        )

        comparison_results.append({
            "clause": clause[:100],
            "spacy_ents": spacy_ents,
            "llm_parties": llm_result.parties,
            "llm_amounts": llm_result.monetary_amounts,
        })

    # Print 3 examples where spaCy wins and 3 where LLM wins
    print("=== 3 cases where spaCy finds entities LLM misses ===")
    spacy_wins = [r for r in comparison_results if len(r["spacy_ents"]) > len(r["llm_parties"])][:3]
    for r in spacy_wins:
        print(f"  spaCy: {r['spacy_ents'][:3]}  |  LLM parties: {r['llm_parties']}")

    print("\n=== 3 cases where LLM extracts what spaCy misses ===")
    llm_wins = [r for r in comparison_results if len(r["llm_amounts"]) > 0 and
                not any(e[1] == "MONEY" for e in r["spacy_ents"])][:3]
    for r in llm_wins:
        print(f"  LLM amounts: {r['llm_amounts']}  |  spaCy MONEY: none found")

    # Recommendation comment:
    # LLM wins for: domain-specific entities (obligation parties, contract roles)
    # spaCy wins for: standard NER (dates, locations, org names) — faster, no API cost

    check([
        (len(spacy_wins) >= 1, "≥ 1 spaCy-win example documented"),
        (len(llm_wins) >= 1,   "≥ 1 LLM-win example documented"),
    ], exercise_id="02b-3")
else:
    print("Install spaCy to complete this challenge.")

In [ ]:
progress()

---
## Readiness checklist before moving to 02c
- [ ] `LegalEntities` parsed for 50 clauses with None handling
- [ ] Batch extraction failure rate documented
- [ ] LLM NER vs spaCy comparison written with examples